In [ ]:
import geopandas as gpd
import osmnx as ox 
from pathlib import Path

In [ ]:
print(Path.cwd())

d:\Wu\2026\Project Portfolio\002 Project\grocery_access


In [5]:
G = ox.load_graphml("data/processed/kootenai_drive.graphml")

In [6]:
stores = gpd.read_file("data/processed/stores_points_with_cities.gpkg")

In [8]:
stores.columns

Index(['element', 'id', 'addr:city', 'addr:housenumber', 'addr:postcode',
       'addr:street', 'name', 'opening_hours', 'shop', 'brand',
       'brand:wikidata', 'official_name', 'addr:state', 'phone', 'website',
       'contact:facebook', 'start_date', 'building', 'designation',
       'addr:country', 'operator', 'operator:wikidata', 'payment:app:walmart',
       'payment:apple_pay', 'payment:google_pay', 'payment:samsung_pay', 'ref',
       'ref:walmart', 'sensory_friendly:accommodation',
       'sensory_friendly:conditional', 'source', 'source:name', 'layer',
       'toilets', 'branch', 'internet_access', 'organic', 'toilets:wheelchair',
       'wheelchair', 'building:levels', 'not:brand:wikidata', 'payment:cash',
       'payment:cheque', 'payment:contactless', 'payment:credit_cards',
       'payment:debit_cards', 'payment:ebt', 'geometry_type', 'index_right',
       'city', 'geometry'],
      dtype='str')

In [12]:
len(stores)

30

In [11]:
stores[["name", "addr:city", "city", "addr:street", "geometry_type", "geometry"]].sort_values(["name"])

,name,addr:city,city,addr:street,geometry_type,geometry
4,Albertsons,Hayden,Hayden,West Prairie Avenue,Point,POINT (515814.247 5288108.75)
5,C & C Grocery,Coeur d'Alene,Coeur d'Alene,NaN,Point,POINT (512703.314 5287894.661)
3,Flour Mill Natural Foods,Hayden,Hayden,West Commerce Drive,Point,POINT (515911.85 5288856.19)
12,Fred Meyer,Coeur d'Alene,Coeur d'Alene,West Kathleen Avenue,Polygon,POINT (515390.535 5284361.564)
26,Gittel's Grocery,Coeur d'Alene,Coeur d'Alene,NaN,Polygon,POINT (516013.751 5281469.361)
0,Grocery Outlet,Coeur d'Alene,Coeur d'Alene,NaN,Point,POINT (515496.942 5283616.399)
28,Grocery Outlet,Post Falls,Post Falls,North Highway 41,Polygon,POINT (508084.904 5286507.542)
11,Harrison Trading Company,Harrison,Harrison,NaN,Polygon,POINT (516148.496 5255651.677)
27,Jordan's Grocery,Coeur d'Alene,Coeur d'Alene,NaN,Polygon,POINT (517671.934 5281126.706)
9,Lakeside Harvest Foods,Coeur d'Alene,Coeur d'Alene,NaN,Polygon,POINT (517393.444 5280087.921)


In [14]:
stores.crs

<Projected CRS: EPSG:26911>
Name: NAD83 / UTM zone 11N
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: North America - between 120°W and 114°W - onshore and offshore. Canada - Alberta; British Columbia; Northwest Territories; Nunavut. United States (USA) - California; Idaho; Nevada, Oregon; Washington.
- bounds: (-120.0, 30.88, -114.0, 83.5)
Coordinate Operation:
- name: UTM zone 11N
- method: Transverse Mercator
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [16]:
# OpenStreeMap uses WGS84 (4326)
stores = stores.to_crs(4326)

In [17]:
stores['x'] = stores.geometry.x
stores['y'] = stores.geometry.y

In [19]:
stores["nearest_node"] = ox.distance.nearest_nodes(
    G,
    stores['x'],
    stores['y']
)

In [20]:
stores[["name", "addr:city", "nearest_node"]]

,name,addr:city,nearest_node
0,Grocery Outlet,Coeur d'Alene,8556346272
1,Little Town Market,Athol,129848349
2,Target,Coeur d'Alene,8638420764
3,Flour Mill Natural Foods,Hayden,129845848
4,Albertsons,Hayden,129776037
5,C & C Grocery,Coeur d'Alene,129721878
6,Safeway,Coeur d'Alene,129865809
7,Walmart Supercenter,Post Falls,130527991
8,Walmart Supercenter,Post Falls,365216968
9,Lakeside Harvest Foods,Coeur d'Alene,129825158


In [21]:
stores.to_file("data/processed/stores_nearest_nodes.gpkg", driver="GPKG")